# 01 - Data Ingestion
Ingest OLA ride-hailing data from ADLS Gen2 (CSV, JSON, Parquet) into bronze Delta tables.

In [ ]:
from pyspark.sql.functions import col, current_timestamp, input_file_name, lit
from pyspark.sql.types import IntegerType, DoubleType, StringType

In [ ]:
configs = {"fs.azure.account.auth.type": "OAuth",
"fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
"fs.azure.account.oauth2.client.id": "",
"fs.azure.account.oauth2.client.secret": '',
"fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/tanent_id/oauth2/token"}


dbutils.fs.mount(
source = "abfss://ola-data@olalakehousedata.dfs.core.windows.net", # container@storageacc
mount_point = "/mnt/ola",
extra_configs = configs)

In [ ]:
%fs
ls "/mnt/ola/raw-data" 

In [ ]:
spark

### Read raw sources — CSV, JSON, Parquet

In [ ]:
customers = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/ola/raw-data/customers/customers.csv")
drivers = spark.read.format("json").load("/mnt/ola/raw-data/drivers/drivers.json")
vehicles = spark.read.format("parquet").load("/mnt/ola/raw-data/vehicles/vehicles.parquet")
locations = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/ola/raw-data/locations/locations.csv")
trips = spark.read.format("json").load("/mnt/ola/raw-data/trips/")

In [ ]:
customers.show()

In [ ]:
customers.printSchema()

In [ ]:
drivers.show()

In [ ]:
drivers.printSchema()

In [ ]:
vehicles.show()

In [ ]:
vehicles.printSchema()

In [ ]:
locations.show()

In [ ]:
locations.printSchema()

In [ ]:
trips.show()

In [ ]:
trips.printSchema()

### Tag each source with ingestion metadata (lineage columns)

In [ ]:
customers = customers.withColumn("_source_file", input_file_name()).withColumn("_ingested_at", current_timestamp())
drivers = drivers.withColumn("_source_file", input_file_name()).withColumn("_ingested_at", current_timestamp())
vehicles = vehicles.withColumn("_source_file", input_file_name()).withColumn("_ingested_at", current_timestamp())
locations = locations.withColumn("_source_file", input_file_name()).withColumn("_ingested_at", current_timestamp())
trips = trips.withColumn("_source_file", input_file_name()).withColumn("_ingested_at", current_timestamp())

### Write to bronze Delta tables

In [ ]:
customers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.bronze.customers")
drivers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.bronze.drivers")
vehicles.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.bronze.vehicles")
locations.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.bronze.locations")
trips.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("ola_lakehouse.bronze.trips")

In [ ]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM ola_lakehouse.bronze.customers
UNION ALL SELECT 'drivers', COUNT(*) FROM ola_lakehouse.bronze.drivers
UNION ALL SELECT 'vehicles', COUNT(*) FROM ola_lakehouse.bronze.vehicles
UNION ALL SELECT 'locations', COUNT(*) FROM ola_lakehouse.bronze.locations
UNION ALL SELECT 'trips', COUNT(*) FROM ola_lakehouse.bronze.trips